You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan Document:

Title: The Future of Artificial Intelligence: Trends, Players, and News

Outline:
I. Introduction
- Definition of Artificial Intelligence
- Importance of AI in various industries
- Preview of key trends and players in the AI field

II. Latest Trends in Artificial Intelligence
A. Machine Learning and Deep Learning advancements
B. Natural Language Processing innovations
C. AI-powered automation and robotics
D. Ethical considerations in AI development

III. Key Players in the AI Industry

I now can give a great answer

Final Answer:

# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction
Artificial Intelligence (AI) is revolutionizing the way we live and work, with its applications spanning across various industries. Defined as the simulation of human intelligence processes by machines, AI has become a crucial tool for businesses looking to streamline processes, improve decision-making, and enhance customer experiences. In this blog post, we will explore the latest trends in AI, key players in the industry, and noteworthy news shaping the future of Artificial Intelligence.

## Latest Trends in Artificial Intelligence
Artificial Intelligence (AI) is continuously evolving, with new trends shaping the landscape of various industries. Machine Learning and Deep Learning have been pivotal in advancing AI, allowing machines to learn from data and make predictions without explicit programming. These technologies have led to breakthroughs in image re

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction
Artificial Intelligence (AI) is revolutionizing the way we live and work, with its applications spanning across various industries. Defined as the simulation of human intelligence processes by machines, AI has become a crucial tool for businesses looking to streamline processes, improve decision-making, and enhance customer experiences. In this blog post, we will explore the latest trends in AI, key players in the industry, and noteworthy news shaping the future of Artificial Intelligence.

## Latest Trends in Artificial Intelligence
Artificial Intelligence (AI) is continuously evolving, with new trends shaping the landscape of various industries. Machine Learning and Deep Learning have been pivotal in advancing AI, allowing machines to learn from data and make predictions without explicit programming. These technologies have led to breakthroughs in image recognition, natural language processing, and autonomous vehicles. Additionally, Natural Language Processing (NLP) has made significant strides in enabling machines to understand and generate human language, revolutionizing how we interact with technology and each other.

AI-powered automation and robotics have also transformed industries by allowing machines to perform tasks that were once exclusive to humans. Ethical considerations have become increasingly important as AI progresses, with concerns around data privacy, algorithm bias, and job displacement requiring attention from organizations and policymakers to ensure responsible development and deployment of AI technologies.

## Key Players in the AI Industry
Leading the way in AI research and development are tech giants like Google, Amazon, and Microsoft, who are driving innovation and shaping the future of AI. Startups are also disrupting the AI landscape with innovative solutions and agile approaches across various sectors. Research institutions, including universities and AI labs, play a crucial role in driving AI innovation through cutting-edge research and collaboration with industry partners, pushing the boundaries of what is possible with artificial intelligence.

## Noteworthy News in Artificial Intelligence
Recent breakthroughs in AI research have opened up new possibilities and advancements in the field, with researchers constantly pushing the boundaries of AI capabilities. Partnerships and collaborations among AI companies, research institutions, and government agencies are accelerating the development and adoption of AI technologies, essential for driving innovation and addressing complex challenges in the AI landscape. Regulatory developments around data privacy, algorithm transparency, and ethical AI are also shaping the future of AI adoption, emphasizing the need for clear guidelines and regulations to ensure responsible and ethical use of AI technologies.

## Conclusion
Staying informed about the latest trends, key players, and news in the AI industry is crucial for professionals, students, decision-makers, and tech enthusiasts. By following our blog for regular updates and insights on the future of AI, you can stay ahead of the curve and make informed decisions about the role of Artificial Intelligence in your personal and professional life. Stay tuned for more updates on Artificial Intelligence!

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [13]:
    topic = "Hospitality"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Hospitality.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: 

Title: Embracing Hospitality Trends: A Guide for Success in the Industry

Outline:

I. Introduction
- Brief overview of the hospitality industry
- Importance of staying updated on trends

II. Latest Trends in Hospitality
- Emphasis on sustainability and eco-friendly practices
- Integration of technology for enhanced guest experiences
- Personalization and customization in service offerings
- Rise of experiential travel and unique accommodations

III. Key Players in the Industry
- Major hotel chains leadi

I now can give a great answer

Final Answer:
# **Embracing Hospitality Trends: A Guide for Success in the Industry**

The hospitality industry is a dynamic and ever-evolving sector that caters to the needs and desires of travelers worldwide. In order to stay competitive and relevant in this fast-paced environment, it is crucial for businesses to embrace the latest trends and innovations. By keeping a pulse on the industry and adapting to changing consumer preferences, hospitality professionals can ensure success and continued growth.

One of the most significant trends in the hospitality industry is the emphasis on sustainability and eco-friendly practices. With increasing awareness of environmental issues, travelers are seeking out hotels and accommodations that prioritize green initiatives. From reducing plastic waste to implementing energy-efficient technologies, sustainability has become a key focus for many businesses in the industry.

Another important trend is the integration of

In [14]:
Markdown(result)

# **Embracing Hospitality Trends: A Guide for Success in the Industry**

The hospitality industry is a dynamic and ever-evolving sector that caters to the needs and desires of travelers worldwide. In order to stay competitive and relevant in this fast-paced environment, it is crucial for businesses to embrace the latest trends and innovations. By keeping a pulse on the industry and adapting to changing consumer preferences, hospitality professionals can ensure success and continued growth.

One of the most significant trends in the hospitality industry is the emphasis on sustainability and eco-friendly practices. With increasing awareness of environmental issues, travelers are seeking out hotels and accommodations that prioritize green initiatives. From reducing plastic waste to implementing energy-efficient technologies, sustainability has become a key focus for many businesses in the industry.

Another important trend is the integration of technology for enhanced guest experiences. From mobile check-in options to in-room smart devices, hotels are leveraging technology to streamline operations and provide guests with personalized services. By embracing digital innovations, hospitality businesses can create a seamless and convenient experience for their customers.

Personalization and customization have also become essential in the hospitality industry. Today's travelers are looking for tailored experiences that cater to their individual preferences and needs. Whether it's personalized room amenities or curated dining options, offering customized services can set businesses apart and create lasting impressions on guests.

In the realm of hospitality, major hotel chains are leading the way in innovation by investing in cutting-edge technologies and sustainable practices. These industry giants are setting the standard for excellence and pushing boundaries to meet the evolving needs of travelers.

On the other hand, boutique hotels are carving out their niche in the market by focusing on personalized experiences and catering to specific demographics. By offering unique amenities and tailored services, boutique hotels are able to create memorable stays for their guests and foster loyalty among their clientele.

Additionally, home-sharing platforms like Airbnb have disrupted the traditional hospitality landscape by offering travelers a more personalized and authentic accommodation experience. By connecting guests with local hosts and providing a home-away-from-home environment, these platforms have revolutionized the way people travel and stay.

The COVID-19 pandemic has had a profound impact on the hospitality industry, forcing businesses to adapt to new health and safety protocols. The shift towards contactless services and enhanced cleaning procedures has become a top priority for hotels and accommodations worldwide, ensuring the safety and well-being of guests and staff.

Furthermore, the growth of staycation and workation trends has emerged as a response to travel restrictions and remote work opportunities. Many people are opting to explore their own cities or work from vacation destinations, creating new opportunities for hospitality businesses to cater to this changing demand.

Business travelers are seeking efficient and comfortable accommodations that cater to their needs while on the road. Providing seamless services and amenities can help hotels attract this valuable demographic and create a loyal customer base.

Leisure travelers, on the other hand, are looking for unique experiences and personalized service that go beyond the traditional offerings. By tapping into this desire for memorable stays, hospitality businesses can differentiate themselves and appeal to a broader audience.

Millennials and Gen Z travelers are increasingly valuing sustainability and social responsibility when choosing their accommodations. By implementing green practices and supporting local communities, hotels can attract eco-conscious travelers and showcase their commitment to ethical tourism.

As the hospitality industry continues to evolve, it is essential for professionals to stay informed on the latest trends and innovations. By keeping a finger on the pulse of the industry and adapting to changing consumer preferences, businesses can thrive and succeed in a competitive marketplace.

I encourage readers to share their own experiences and insights on hospitality trends, as well as explore the resources provided for further reading. By learning from industry experts and staying up-to-date on best practices, hospitality professionals can position themselves for success and growth in the ever-changing landscape of the industry.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```